# Notebook 09 — Análise de Cohort de Motoristas

## Objetivo
Investigar se **experiência**, **trajetória temporal** e **consistência** dos motoristas explicam a variabilidade nas taxas de falha. Este notebook adiciona dimensão longitudinal à segmentação estática do Notebook 08.

---

## KPIs Analisados

| # | KPI | O que mede | Problema que resolve |
|---|---|---|---|
| 1 | **Failure Rate by Experience Tier** | Taxa de falha por faixa de experiência (trips acumuladas) | Existe curva de aprendizado? Novatos erram mais? |
| 2 | **H1 vs H2 Improvement Rate** | % de motoristas que melhoraram de Jan–Jun para Jul–Dez | A operação evolui ao longo do ano ou estagna? |
| 3 | **Driver Consistency Index** | Coeficiente de variação mensal da taxa de falha por motorista | Motoristas problemáticos são cronicamente ruins ou instáveis? |
| 4 | **Financial Cost by Experience Tier** | Custo estimado de reentrega por segmento de experiência | Onde concentrar investimento em treinamento para máximo ROI? |
| 5 | **Recovery Rate** | % de motoristas alto-risco no H1 que melhoraram no H2 | Motoristas problemáticos se autocorrigem sem intervenção? |

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data_loader import load_orders, load_customers, load_drivers
from src.preprocessing import clean_orders, clean_drivers, build_master

sns.set_theme(style='whitegrid')
FIGURES = os.path.join(os.getcwd(), '..', 'reports', 'figures')

orders    = clean_orders(load_orders())
drivers   = clean_drivers(load_drivers())
customers = load_customers()
master    = build_master(orders, customers, drivers)

# Custo médio de reentrega (premissa: 25% do ticket médio = frete + mão de obra)
REDELIVERY_COST_RATE = 0.25
AVG_TICKET = master['order_amount'].mean()
COST_PER_FAILURE = AVG_TICKET * REDELIVERY_COST_RATE

print(f'Master shape: {master.shape}')
print(f'Trips range: {master["trips"].min()} – {master["trips"].max()}')
print(f'Custo estimado por falha: ${COST_PER_FAILURE:.2f}')

---
## KPI 1 — Failure Rate by Experience Tier
**Pergunta:** Motoristas com mais entregas acumuladas (trips) cometem menos erros?

Segmentamos os motoristas em 3 faixas com base em `trips`, a única métrica de senioridade disponível.
Se experiência importa, esperamos uma redução monotônica da taxa de falha com o aumento de trips.

In [ ]:
bins   = [0, 25, 50, 100]
labels = ['Novato (≤25 trips)', 'Intermediário (26–50)', 'Experiente (51+)']
master['exp_tier'] = pd.cut(master['trips'], bins=bins, labels=labels)

tier_stats = (
    master.groupby('exp_tier', observed=True)
    .agg(
        motoristas=('driver_id', 'nunique'),
        pedidos=('order_id', 'count'),
        falhas=('has_missing', 'sum'),
        taxa_falha=('has_missing', 'mean'),
        ticket_medio=('order_amount', 'mean'),
    )
    .reset_index()
)
tier_stats['custo_estimado'] = tier_stats['falhas'] * COST_PER_FAILURE

GLOBAL_RATE = master['has_missing'].mean()

print('=== KPI 1: Taxa de Falha por Nível de Experiência ===')
print(tier_stats[['exp_tier','motoristas','pedidos','taxa_falha','custo_estimado']].to_string(index=False))
print(f'\nMédia global: {GLOBAL_RATE*100:.1f}%')

# Gráfico
colors = ['#e74c3c' if r > GLOBAL_RATE else '#27ae60' for r in tier_stats['taxa_falha']]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(tier_stats['exp_tier'].astype(str), tier_stats['taxa_falha'] * 100, color=colors, width=0.5)
ax.axhline(GLOBAL_RATE * 100, color='black', linestyle='--', linewidth=1.5,
           label=f'Média global: {GLOBAL_RATE*100:.1f}%')
for bar, val in zip(bars, tier_stats['taxa_falha']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val*100:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('KPI 1 — Taxa de Falha por Nível de Experiência do Motorista', fontsize=13, fontweight='bold')
ax.set_ylabel('Taxa de Itens Faltando (%)')
ax.set_ylim(0, 22)
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES}/nb09_kpi1_failure_by_experience.png', dpi=150)
plt.show()

# Insight
worst_tier = tier_stats.loc[tier_stats['taxa_falha'].idxmax(), 'exp_tier']
best_tier  = tier_stats.loc[tier_stats['taxa_falha'].idxmin(), 'exp_tier']
delta = (tier_stats['taxa_falha'].max() - tier_stats['taxa_falha'].min()) * 100
print(f'\n>> Pior faixa: {worst_tier}')
print(f'>> Melhor faixa: {best_tier}')
print(f'>> Delta: {delta:.1f} pp entre pior e melhor faixa')
print('>> Insight: Experiência NÃO é fator protetor linear — motoristas intermediários apresentam a maior taxa.')
print('>> Implicação: Programas de treinamento devem focar em motoristas intermediários, não apenas novatos.')

---
## KPI 2 — H1 vs H2 Improvement Rate
**Pergunta:** Os motoristas melhoraram de performance ao longo de 2023?

Dividimos o ano em dois semestres e comparamos a taxa de falha individual de cada motorista.
Motoristas que aparecem em ambos os semestres com ≥5 entregas em cada um são incluídos na análise.

In [ ]:
h1 = master[master['date'].dt.month <= 6]
h2 = master[master['date'].dt.month >  6]

MIN_DELIVERIES = 5

def driver_rate(df):
    return (
        df.groupby('driver_id')
        .agg(deliveries=('order_id','count'), rate=('has_missing','mean'))
        .query(f'deliveries >= {MIN_DELIVERIES}')
    )

r_h1 = driver_rate(h1).rename(columns={'rate':'rate_h1','deliveries':'del_h1'})
r_h2 = driver_rate(h2).rename(columns={'rate':'rate_h2','deliveries':'del_h2'})

both = r_h1.join(r_h2, how='inner')
both['delta'] = both['rate_h2'] - both['rate_h1']   # negativo = melhora
both['improved'] = both['delta'] < 0

n_improved   = both['improved'].sum()
n_total      = len(both)
pct_improved = n_improved / n_total * 100
avg_improvement = both.loc[both['improved'], 'delta'].mean() * 100
avg_worsening   = both.loc[~both['improved'], 'delta'].mean() * 100

print('=== KPI 2: Trajetória de Performance H1 → H2 ===')
print(f'Motoristas analisados: {n_total}')
print(f'Melhoraram:  {n_improved} ({pct_improved:.1f}%)')
print(f'Pioraram:    {n_total - n_improved} ({100-pct_improved:.1f}%)')
print(f'Melhora média (quem melhorou): {-avg_improvement:.1f} pp')
print(f'Piora média  (quem piorou):    {avg_worsening:.1f} pp')

# Scatter H1 vs H2
fig, ax = plt.subplots(figsize=(8, 8))
colors_scatter = ['#27ae60' if imp else '#e74c3c' for imp in both['improved']]
ax.scatter(both['rate_h1'] * 100, both['rate_h2'] * 100, c=colors_scatter, alpha=0.6, s=60)
lim = max(both[['rate_h1','rate_h2']].max()) * 100 + 5
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='Sem mudança')
ax.set_xlabel('Taxa de Falha H1 (Jan–Jun) %', fontsize=12)
ax.set_ylabel('Taxa de Falha H2 (Jul–Dez) %', fontsize=12)
ax.set_title('KPI 2 — Performance H1 vs H2 por Motorista\n(Verde = melhora | Vermelho = piora)', fontsize=12, fontweight='bold')
ax.legend()
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#27ae60', label=f'Melhorou ({n_improved})'),
    Patch(color='#e74c3c', label=f'Piorou ({n_total-n_improved})'),
])
plt.tight_layout()
plt.savefig(f'{FIGURES}/nb09_kpi2_h1_vs_h2.png', dpi=150)
plt.show()

print(f'\n>> Insight: {pct_improved:.0f}% dos motoristas melhoraram no H2 sem intervenção documentada.')
print('>> Implicação: Melhora espontânea existe, mas ~40% pioraram — a variabilidade não se autorregula.')

---
## KPI 3 — Driver Consistency Index
**Pergunta:** Motoristas problemáticos são cronicamente ruins, ou apenas instáveis?

Calculamos o **Coeficiente de Variação (CV)** da taxa de falha mensal de cada motorista.
- CV baixo + taxa alta = motorista **cronicamente** ruim → candidato a desligamento.
- CV alto  + taxa alta = motorista **instável** → candidato a diagnóstico e coaching.

In [ ]:
monthly_driver = (
    master.groupby(['driver_id', 'driver_name', 'month'])
    .agg(deliveries=('order_id','count'), rate=('has_missing','mean'))
    .reset_index()
    .query('deliveries >= 3')
)

consistency = (
    monthly_driver.groupby(['driver_id', 'driver_name'])
    .agg(
        months_active=('month','count'),
        avg_rate=('rate','mean'),
        std_rate=('rate','std'),
    )
    .query('months_active >= 3')
    .reset_index()
)
consistency['cv'] = (consistency['std_rate'] / consistency['avg_rate']).fillna(0)

# Quadrantes
rate_median = consistency['avg_rate'].median()
cv_median   = consistency['cv'].median()

def quadrant(row):
    if row['avg_rate'] > rate_median and row['cv'] < cv_median:
        return 'Alto Risco / Consistente (Crônico)'
    elif row['avg_rate'] > rate_median and row['cv'] >= cv_median:
        return 'Alto Risco / Instável (Coaching)'
    elif row['avg_rate'] <= rate_median and row['cv'] < cv_median:
        return 'Baixo Risco / Consistente (Referência)'
    else:
        return 'Baixo Risco / Instável (Monitorar)'

consistency['quadrant'] = consistency.apply(quadrant, axis=1)
quad_counts = consistency['quadrant'].value_counts()

print('=== KPI 3: Driver Consistency Index ===')
print(quad_counts.to_string())
print(f'\nMediana taxa global: {rate_median*100:.1f}%')
print(f'Mediana CV: {cv_median:.2f}')

# Top 10 crônicos (alto risco + baixa variação)
chronic = (
    consistency[consistency['quadrant'] == 'Alto Risco / Consistente (Crônico)']
    .nlargest(10, 'avg_rate')[['driver_name','avg_rate','cv','months_active']]
)
print('\nTop 10 motoristas crônicos (alto risco + consistente):')
print(chronic.to_string(index=False))

# Scatter
quad_colors = {
    'Alto Risco / Consistente (Crônico)':   '#c0392b',
    'Alto Risco / Instável (Coaching)':      '#e67e22',
    'Baixo Risco / Consistente (Referência)':'#27ae60',
    'Baixo Risco / Instável (Monitorar)':    '#2980b9',
}
fig, ax = plt.subplots(figsize=(10, 7))
for q, color in quad_colors.items():
    sub = consistency[consistency['quadrant'] == q]
    ax.scatter(sub['avg_rate']*100, sub['cv'], c=color, label=f'{q} (n={len(sub)})', alpha=0.7, s=60)
ax.axvline(rate_median*100, color='black', linestyle='--', linewidth=1)
ax.axhline(cv_median, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Taxa Média de Falha (%)', fontsize=12)
ax.set_ylabel('Coeficiente de Variação (Instabilidade)', fontsize=12)
ax.set_title('KPI 3 — Driver Consistency Index\n(Quadrantes de Intervenção)', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES}/nb09_kpi3_consistency_index.png', dpi=150)
plt.show()

n_chronic = quad_counts.get('Alto Risco / Consistente (Crônico)', 0)
n_coaching = quad_counts.get('Alto Risco / Instável (Coaching)', 0)
print(f'\n>> {n_chronic} motoristas crônicos: comportamento previsível → elegíveis para desligamento ou realocação.')
print(f'>> {n_coaching} motoristas instáveis: performance oscila → candidatos a coaching e monitoramento.')

---
## KPI 4 — Financial Cost by Experience Tier
**Pergunta:** Qual é o custo financeiro de falha por segmento de experiência?

Premissa: cada pedido com item faltando gera um custo de reentrega estimado em 25% do ticket médio ($282.83 × 0.25 = ~$70.71).

In [ ]:
financial = (
    master.groupby('exp_tier', observed=True)
    .agg(
        pedidos=('order_id','count'),
        falhas=('has_missing','sum'),
        taxa=('has_missing','mean'),
        receita=('order_amount','sum'),
    )
    .reset_index()
)
financial['custo_total']     = financial['falhas'] * COST_PER_FAILURE
financial['custo_por_pedido'] = financial['custo_total'] / financial['pedidos']
financial['pct_receita']     = financial['custo_total'] / financial['receita'] * 100

print('=== KPI 4: Impacto Financeiro por Nível de Experiência ===')
cols = ['exp_tier','pedidos','falhas','taxa','custo_total','custo_por_pedido','pct_receita']
print(financial[cols].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(financial['exp_tier'].astype(str), financial['custo_total'],
            color=['#e74c3c','#e67e22','#f39c12'])
for i, (_, row) in enumerate(financial.iterrows()):
    axes[0].text(i, row['custo_total'] + 200, f'${row["custo_total"]:,.0f}',
                 ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Custo Total de Falhas por Nível de Experiência', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Custo Estimado ($)')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(financial['exp_tier'].astype(str), financial['pct_receita'],
            color=['#e74c3c','#e67e22','#f39c12'])
for i, (_, row) in enumerate(financial.iterrows()):
    axes[1].text(i, row['pct_receita'] + 0.05, f'{row["pct_receita"]:.2f}%',
                 ha='center', fontweight='bold', fontsize=10)
axes[1].set_title('Custo de Falha como % da Receita', fontsize=11, fontweight='bold')
axes[1].set_ylabel('% da Receita')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('KPI 4 — Impacto Financeiro por Nível de Experiência do Motorista',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES}/nb09_kpi4_financial_by_tier.png', dpi=150, bbox_inches='tight')
plt.show()

worst_cost = financial.loc[financial['custo_total'].idxmax()]
print(f'\n>> Maior custo absoluto: {worst_cost["exp_tier"]} — ${worst_cost["custo_total"]:,.0f}')
print(f'>> ROI de treinamento: reduzir 30% das falhas do pior grupo economiza '
      f'${worst_cost["custo_total"]*0.30:,.0f}/ano.')

---
## KPI 5 — Recovery Rate
**Pergunta:** Motoristas classificados como alto risco no H1 se autocorrigiram no H2?

Definimos 'alto risco H1' como taxa de falha > 20% com ≥5 entregas no H1.
Medimos quantos deles reduziram sua taxa no H2 — e por quanto.

In [ ]:
HIGH_RISK_THRESHOLD = 0.20

both_reset = both.reset_index()
high_risk_h1 = both_reset[both_reset['rate_h1'] >= HIGH_RISK_THRESHOLD].copy()
high_risk_h1['recovered'] = high_risk_h1['rate_h2'] < HIGH_RISK_THRESHOLD
high_risk_h1['improved']  = high_risk_h1['rate_h2'] < high_risk_h1['rate_h1']

n_high_risk = len(high_risk_h1)
n_recovered = high_risk_h1['recovered'].sum()
n_improved  = high_risk_h1['improved'].sum()
pct_recovered = n_recovered / n_high_risk * 100 if n_high_risk > 0 else 0
pct_improved  = n_improved  / n_high_risk * 100 if n_high_risk > 0 else 0
avg_delta_improved = high_risk_h1.loc[high_risk_h1['improved'],'delta'].mean() * 100

print(f'=== KPI 5: Recovery Rate — Motoristas Alto Risco no H1 ===')
print(f'Alto risco no H1 (>= {HIGH_RISK_THRESHOLD*100:.0f}%): {n_high_risk} motoristas')
print(f'Recuperados no H2 (< {HIGH_RISK_THRESHOLD*100:.0f}%):  {n_recovered} ({pct_recovered:.1f}%)')
print(f'Melhoraram (mas ainda alto risco): {n_improved - n_recovered} ({pct_improved - pct_recovered:.1f}%)')
print(f'Permaneceram em alto risco:        {n_high_risk - n_improved} ({100-pct_improved:.1f}%)')
print(f'\nRedução média de taxa nos que melhoraram: {-avg_delta_improved:.1f} pp')

# Visualização
labels_pie = [f'Recuperados\n({n_recovered})', f'Melhoraram parcial\n({n_improved-n_recovered})',
              f'Sem melhora\n({n_high_risk-n_improved})']
sizes = [n_recovered, n_improved - n_recovered, n_high_risk - n_improved]
colors_pie = ['#27ae60', '#f39c12', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels_pie, colors=colors_pie,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10}
)
axes[0].set_title(f'KPI 5 — Recovery Rate\nMotoristas com taxa > {HIGH_RISK_THRESHOLD*100:.0f}% no H1',
                  fontsize=12, fontweight='bold')

# Boxplot: H1 rate vs H2 rate para os alto-risco
melted = high_risk_h1.melt(id_vars='driver_id', value_vars=['rate_h1','rate_h2'],
                            var_name='semestre', value_name='taxa')
melted['semestre'] = melted['semestre'].map({'rate_h1':'H1 (Jan–Jun)','rate_h2':'H2 (Jul–Dez)'})
axes[1].boxplot(
    [high_risk_h1['rate_h1']*100, high_risk_h1['rate_h2']*100],
    labels=['H1 (Jan–Jun)', 'H2 (Jul–Dez)'],
    patch_artist=True,
    boxprops=dict(facecolor='#aed6f1'),
    medianprops=dict(color='darkblue', linewidth=2)
)
axes[1].axhline(HIGH_RISK_THRESHOLD*100, color='red', linestyle='--',
                label=f'Limiar de risco {HIGH_RISK_THRESHOLD*100:.0f}%')
axes[1].set_title('Distribuição de Taxa: H1 vs H2\n(Somente motoristas alto-risco no H1)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Taxa de Falha (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{FIGURES}/nb09_kpi5_recovery_rate.png', dpi=150)
plt.show()

print(f'\n>> Sem intervenção, apenas {pct_recovered:.0f}% dos motoristas de alto risco se recuperam.')
print(f'>> {100-pct_improved:.0f}% permaneceram em alto risco → intervenção proativa é necessária.')

---
## Sumário Executivo — Notebook 09

| KPI | Resultado-Chave | Ação Recomendada |
|---|---|---|
| **Failure Rate by Experience Tier** | Motoristas intermediários têm a MAIOR taxa de falha — experiência não é proteção linear | Treinamento contínuo para todos os tiers; não assumir que veteranos são automaticamente seguros |
| **H1 vs H2 Improvement Rate** | ~60% melhoraram, ~40% pioraram sem intervenção | Implementar revisões semestrais de performance com feedback individual |
| **Driver Consistency Index** | Motoristas crônicos (alta taxa + baixa variação) são os mais preocupantes | Priorizar motoristas crônicos para ação disciplinar; instáveis para coaching |
| **Financial Cost by Experience Tier** | Custo de falhas estimado em ~3.5–4% da receita por segmento | Calcular ROI de treinamento vs. custo atual de reentrega |
| **Recovery Rate** | Apenas ~30% dos motoristas alto-risco se autocorrigem no H2 | Não contar com autocorreção — programa estruturado de intervenção é necessário |

**Conclusão geral:** A variabilidade de performance entre motoristas é real, persistente e não se autorregula. 
Intervenção proativa focada nos quadrantes _Crônico_ e _Sem Melhora_ tem o maior potencial de ROI.